# Restaurant assistant agent demo
The restaurant assistant agent brings together number for Amazon Bedrock Agentcore capabilities such as Runtime, Short-term memory and Long-term memory to demonstrate integrated Observability capabilities. The demo uses Strands framework for constructing the AI agent.

#### Tutorial details
| Information         | Details                                                  |
|---------------------|----------------------------------------------------------|
| tutorial type       | conversational                                           |
| Agent type          | Single                                                   |
| tutorial components | Strands, Runtime, memory, observability                  |
| tutorial vertical   | General                                                  |
| Example complexity  | Intermediate                                             |
| SDK used            | Amazon Bedrock AgentCore SDK, boto3, Strands             |


## Prerequisites

* Python 3.10+
* UV package manager
* AWS credentials
* Amazon Bedrock model access

#### Before starting initialize Python virtual environment
```
uv venv --python 3.13
source .venv/bin/activate
python -m ensurepip --default-pip
```

In [ ]:
%pip install -r requirements-dev.txt --quiet
%pip install -r requirements.txt --quiet

## Tutorial step-by-step

#### Update environment variables and generate `.env` file

In [ ]:
%%writefile .env
# =============================================================================
# AWS configurations
# =============================================================================
AWS_REGION="us-east-1"

# =============================================================================
# Amazon Bedrock configurations
# =============================================================================
BEDROCK_MODEL_ID = us.anthropic.claude-sonnet-4-5-20250929-v1:0


#### Create Strands agent
The following strands agent leverages AgentCore memory to recall user preferences before generating a response.

In [ ]:
%%writefile restaurant_strands_agent.py
"""Strands Culinary Assistant"""
import os
import uuid
from ddgs import DDGS
from strands import Agent, tool
from strands.models import BedrockModel
from strands.telemetry import StrandsTelemetry
from strands_tools.agent_core_memory import AgentCoreMemoryToolProvider
# from bedrock_agentcore.memory.integrations.strands.config import (
#     AgentCoreMemoryConfig,
#     RetrievalConfig,
# )
# from bedrock_agentcore.memory.integrations.strands.session_manager import (
#     AgentCoreMemorySessionManager,
# )
from bedrock_agentcore.runtime import BedrockAgentCoreApp, RequestContext
from opentelemetry import baggage, context


SYSTEM_PROMPT = """
You are the Culinary Assistant, a sophisticated restaurant recommendation assistant.
PURPOSE:
- Help users discover restaurants based on their preferences
- Remember user preferences throughout the conversation
- Provide upto 3 personalized dining recommendations
- Include street address for the restaurant recommendation

You have access to a Memory tool that enables you to:
- Retrieve previously stored information to personalize recommendations

You have access to a web search tool that enables you to:
- Retrieve street address of the 

"""
MODEL_ID = os.getenv("BEDROCK_MODEL_ID")
MEMORY_ID = os.getenv("BEDROCK_AGENTCORE_MEMORY_ID")
REGION = os.getenv("AWS_REGION", "us-east-1")

# Initialize Strands telemetry for 3P
if os.getenv("DISABLE_ADOT_OBSERVABILITY"):
    strands_telemetry = StrandsTelemetry()
    strands_telemetry.setup_otlp_exporter()

# Initialize agent application wrapper
app = BedrockAgentCoreApp()


@tool
def web_search(query: str) -> str:
    """
    Search the web for information using DuckDuckGo.

    Args:
        query: The search query

    Returns:
        A string containing the search results
    """
    try:
        ddgs = DDGS()
        results = ddgs.text(query, max_results=5)

        formatted_results = []
        for i, result in enumerate(results, 1):
            formatted_results.append(
                f"{i}. {result.get('title', 'No title')}\n"
                f"   {result.get('body', 'No summary')}\n"
                f"   Source: {result.get('href', 'No URL')}\n"
            )

        return "\n".join(formatted_results) if formatted_results else "No results found."

    except Exception as e:
        return f"Error searching the web: {str(e)}"


def set_session_context(session_id):
    """Session ID OpenTelemetry baggage for distributed trace correlation"""
    ctx = baggage.set_baggage("session.id", session_id)
    token = context.attach(ctx)
    print(f"Session ID context baggage '{session_id}' attached")
    return token


def initialize_agent(req_context: RequestContext):
    """Initialize the agent with memory tools"""

    # actor_id = 'default_user'  # 'user-20251103170743'
    headers = getattr(req_context, "request_headers", {}) or {}
    actor_id = headers.get(
        # "x-amzn-bedrock-agentcore-runtime-custom-actorid",
        'X-Amzn-Bedrock-AgentCore-Runtime-User-Id',
        'default_actor'
    )
    session_id = getattr(req_context, "session_id", str(uuid.uuid4()))
    print(f"actor: {actor_id}, session: {session_id}")
    set_session_context(session_id)
    
    model = BedrockModel(
        model_id=MODEL_ID,
    )

    memory_provider = AgentCoreMemoryToolProvider(
        memory_id=MEMORY_ID,
        actor_id=actor_id,
        session_id=session_id,
        namespace=f"/users/{actor_id}/preferences",
        region=REGION
    )
    agent = Agent(
        tools=[web_search] + memory_provider.tools,
        model=model,
        system_prompt=SYSTEM_PROMPT
    )

    # config = AgentCoreMemoryConfig(
    #     memory_id=MEMORY_ID,
    #     session_id=session_id,
    #     actor_id=actor_id,
    #     retrieval_config={
    #         "/users/{actorId}/preferences": RetrievalConfig(
    #             top_k=5,
    #             relevance_score=0.7
    #         )
    #     }
    # )
    # session_manager = AgentCoreMemorySessionManager(config, region_name=REGION)
    # agent = Agent(model=model, session_manager=session_manager)

    return agent


@app.entrypoint
def strands_agent_bedrock(payload, context: RequestContext):
    """
    Invoke the agent with a payload
    """

    user_input = payload.get("prompt")
    print("User input:", user_input)

    agent = initialize_agent(context)
    response = agent(user_input)
    return response.message['content'][0]['text']


if __name__ == "__main__":
    app.run()


#### Configure AgentCore Runtime for deployment

Next we will use our starter toolkit to configure the AgentCore Runtime deployment with an entrypoint, the execution role we just created and a requirements file. We will also configure the starter kit to auto create the Amazon ECR repository on launch.

During the configure step, your docker file will be generated based on your application code. 

Please note that when using the `bedrock_agentcore_starter_toolkit` to configure your agent, it takes care of the opentelemetry instrumentation. 

In [ ]:
from dotenv import dotenv_values
from bedrock_agentcore_starter_toolkit import Runtime

config = dotenv_values(".env")
region = config.get("AWS_REGION", "us-east-1")

agentcore_runtime = Runtime()
agent_name = "restaurant_assistant_demo"
response = agentcore_runtime.configure(
    entrypoint="restaurant_strands_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name=agent_name,
    memory_mode='STM_AND_LTM',
    # disable_otel=True
)
response

### Deploy to AgentCore Runtime

Now that we've got a docker file, let's launch the agent to the AgentCore Runtime. This will create the Amazon ECR repository and the AgentCore Runtime

In [ ]:
import base64
from dotenv import dotenv_values
from bedrock_agentcore_starter_toolkit import Runtime

config = dotenv_values(".env")

# Bedrock AgentCore configuration
region = config.get("AWS_REGION", "us-east-1")
model_id = config.get("BEDROCK_MODEL_ID", "us.anthropic.claude-sonnet-4-5-20250929-v1:0")

# Langfuse configuration
# otel_endpoint = config.get("LANGFUSE_OTEL_ENDPOINT", "https://us.cloud.langfuse.com/api/public/otel")
# langfuse_secret_key = config.get("LANGFUSE_SECRET_KEY", "")  # For production key should be securely stored
# langfuse_public_key = config.get("LANGFUSE_PUBLIC_KEY", "")  # For production key should be securely stored
# langfuse_auth_token = base64.b64encode(f"{langfuse_public_key}:{langfuse_secret_key}".encode()).decode()
# otel_auth_header = f"Authorization=Basic {langfuse_auth_token}"

# Braintrust configuration
# otel_endpoint = config.get("BRAINTRUST_OTEL_ENDPOINT", "https://api.braintrust.dev/otel")
# braintrust_api_key = config.get("BRAINTRUST_API_KEY", "")  # For production key should be securely stored
# braintrust_project_id = config.get("BRAINTRUST_PROJECT_ID", "")
# otel_auth_header = f"Authorization=Bearer {braintrust_api_key}, x-bt-parent=project_id:{braintrust_project_id}"

agentcore_runtime = Runtime()
launch_result = agentcore_runtime.launch(
    auto_update_on_conflict=True,
    env_vars={
        "BEDROCK_MODEL_ID": model_id,
        "AWS_REGION": region,
        # "DISABLE_ADOT_OBSERVABILITY": "true",
        # "OTEL_EXPORTER_OTLP_ENDPOINT": otel_endpoint,  # Use Langfuse OTEL endpoint
        # "OTEL_EXPORTER_OTLP_HEADERS": otel_auth_header,  # Add Langfuse OTEL auth header
    }
)
launch_result

### Check Deployment Status

Wait for the runtime and memory to be ready before invoking:

In [ ]:
import time
status_response = agentcore_runtime.status()
status = status_response.endpoint['status']
end_status = ['READY', 'CREATE_FAILED', 'DELETE_FAILED', 'UPDATE_FAILED']
while status not in end_status:
    time.sleep(10)
    status_response = agentcore_runtime.status()
    status = status_response.endpoint['status']
    print(status)
status

## Saving Previous Conversations to Memory

In this section, we'll demonstrate how to hydrate the short-term memory, which automatically triggers the long-term memory extraction process behind the scenes.

### Hydrating Short-Term Memory

When we save conversations to a memory resource configured with extraction strategies, the system automatically processes this information for long-term retention without requiring additional code.

> Note: Run the following code multiple times utill you see the LTM memory retrieved.

In [ ]:
import yaml
import uuid
from bedrock_agentcore.memory import MemoryClient

# Read agent manifest
with open('.bedrock_agentcore.yaml', 'r') as f:
    agent_manifest = yaml.safe_load(f)

agent_name = agent_manifest["default_agent"]
region = agent_manifest["agents"][agent_name]["aws"]["region"]
memory_id = agent_manifest["agents"][agent_name]["memory"]["memory_id"]
print(f"Memory id: {memory_id}")

actor_id = "default_actor"
session_id = "foodie-example-session-20251123"
# Note: ensure namespace pattern exactly matches the LTM user preference strategy
namespace = f"/users/{actor_id}/preferences"
print(f"User preference namespace: {namespace}")

memory_client = MemoryClient(region_name=region)
memory_events = memory_client.list_events(
    memory_id=memory_id,
    actor_id=actor_id,
    session_id=session_id,
    max_results=5
)
if not memory_events:
    print("\nHydrating short term memory with previous conversations...")
    previous_messages = [
        ("Hi, I'm John", "USER"),
        ("Hi John, how can I help you with food recommendations today?", "ASSISTANT"),
        ("I'm looking for some vegetarian dishes to try this weekend.", "USER"),
        ("That sounds great! I'd be happy to help with vegetarian recommendations. Do you have any specific ingredients or cuisine types you prefer?", "ASSISTANT"),
        ("Yes, I really like tofu and fresh vegetables in my dishes", "USER"),
        ("Perfect! Tofu and fresh vegetables make for excellent vegetarian meals. I can suggest some stir-fries, Buddha bowls, or tofu curries. Do you have any other preferences?", "ASSISTANT"),
        ("I also really enjoy Italian cuisine. I love pasta dishes and would like them to be vegetarian-friendly.", "USER"),
        ("Excellent! Italian cuisine has wonderful vegetarian options. I can recommend pasta primavera, mushroom risotto, eggplant parmesan, or penne arrabbiata. The combination of Italian flavors with vegetarian ingredients creates delicious meals!", "ASSISTANT"),
        ("I spent 2 hours looking through cookbooks but couldn't find inspiring vegetarian Italian recipes", "USER"),
        ("I'm sorry you had trouble finding inspiring recipes! Let me help you with some creative vegetarian Italian dishes. How about stuffed bell peppers with Italian herbs and rice, spinach and ricotta cannelloni, or a Mediterranean vegetable lasagna?", "ASSISTANT"),
        ("Hey, I appreciate food assistants with good taste", "USER"),
        ("Ha! I definitely try to bring good taste to the table! Speaking of which, shall we explore some more vegetarian Italian recipes that might inspire you?", "ASSISTANT")
    ]
    # Save the conversation history to long-term memory
    initial = memory_client.create_event(
        memory_id=memory_id,
        actor_id=actor_id,
        session_id=session_id,
        messages=previous_messages,
    )
    print("✓ Conversation saved in memory")
else:
    try:
        # Query the memory system for food preferences
        food_preferences = memory_client.retrieve_memories(
            memory_id=memory_id,
            namespace=namespace,
            query="food preferences",
            top_k=3  # Return up to 3 most relevant results
        )

        if food_preferences:
            print(f"Retrieved {len(food_preferences)} relevant preference records:")
            for i, record in enumerate(food_preferences):
                print(f"\nMemory {i+1}:")
                print(f"- Content: {record.get('content', 'Not specified')}")
        else:
            print("No matching preference records found.")

    except Exception as e:
        print(f"Error retrieving preference records: {e}")



#### [Optional] Searching LTM for user preference using boto3

In [ ]:
import boto3
bedrock_agent_core_client = boto3.client(
    "bedrock-agentcore",
    region_name=region
)

params = {"memoryId": memory_id, "namespace": namespace, "searchCriteria": {"searchQuery": "food preferences"}}

resp_recs = bedrock_agent_core_client.retrieve_memory_records(**params)
resp_recs

### Invoking AgentCore Runtime

Finally, we can invoke our AgentCore Runtime with a payload

In [ ]:
import json
invoke_response = agentcore_runtime.invoke(
    {"prompt": "I am visiting las vegas. Can you recommend restaurants based on my food preferences"}
)
from IPython.display import Markdown, display
display(Markdown(json.loads("".join(invoke_response['response']))))

### Invoking AgentCore Runtime with boto3

Now that your AgentCore Runtime was created you can invoke it with any AWS SDK. For instance, you can use the boto3 `invoke_agent_runtime` method for it.

In [ ]:
import boto3
import yaml
import json
from IPython.display import Markdown, display

QUERY = "I am visiting las vegas. Can you recommend restaurants based on my food preferences"

# Read agent manifest
with open('.bedrock_agentcore.yaml', 'r') as f:
    agent_manifest = yaml.safe_load(f)

agent_name = agent_manifest["default_agent"]
region = agent_manifest["agents"][agent_name]["aws"]["region"]
agent_arn = agent_manifest["agents"][agent_name]["bedrock_agentcore"]["agent_arn"]

agentcore_client = boto3.client(
    'bedrock-agentcore',
    region_name=region
)

boto3_response = agentcore_client.invoke_agent_runtime(
    agentRuntimeArn=agent_arn,
    qualifier="DEFAULT",
    payload=json.dumps({"prompt": QUERY})
)
if "text/event-stream" in boto3_response.get("contentType", ""):
    content = []
    for line in boto3_response["response"].iter_lines(chunk_size=1):
        if line:
            line = line.decode("utf-8")
            if line.startswith("data: "):
                line = line[6:]
                print(line)
                content.append(line)
    display(Markdown("\n".join(content)))
else:
    try:
        events = []
        for event in boto3_response.get("response", []):
            events.append(event)
    except Exception as e:
        events = [f"Error reading EventStream: {e}"]
    display(Markdown(json.loads(events[0].decode("utf-8"))))

## Cleanup instructions

Don't forget to provide the cleanup instructions for any resources created

In [ ]:
from bedrock_agentcore_starter_toolkit import Runtime
agentcore_runtime = Runtime()
destroy_result = agentcore_runtime.destroy()